# Week 7: K-Nearest Neighbors & Distance Measures

## Student Practice Notebook

**Name:** Preetham Manchikanti
**Roll Number:** AP24110011315
**Date:** 15th September 2026

## Learning Objectives

After completing this lab, you will be able to:

- Compute Euclidean and Manhattan distance between two points, by hand and in code
- Explain how KNN classifies a new point using its nearest neighbors
- Fit a `KNeighborsClassifier` correctly (split → scale on train only → fit)
- Explain why K matters, and identify overfitting/underfitting as K changes
- Solve a KNN numerical problem by hand — the kind you'll see in a written exam

### Why this week matters
Every model you've built so far (Weeks 5-6) assumed an equation. KNN assumes nothing — it just asks "what do my nearest neighbors look like?" It's the simplest possible classifier, and understanding *why* it works (and where it breaks) builds the intuition you'll need for every distance-based method later in the course.


## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

%matplotlib inline
np.random.seed(1)


In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
df['species'] = df['target'].map(dict(enumerate(iris.target_names)))
df.head()


## Part 1 — Distance Measures

KNN's entire decision rule is "find the closest points." "Closest" needs a precise definition — that's what a distance metric gives us.

### 1.1 Demo: Euclidean and Manhattan distance, by hand


In [ ]:
p1 = np.array([5.1, 3.5, 1.4, 0.2])   # a sample flower's 4 measurements
p2 = np.array([6.4, 3.2, 4.5, 1.5])   # a different flower

euclidean = np.sqrt(np.sum((p1 - p2) ** 2))
manhattan = np.sum(np.abs(p1 - p2))

print('Euclidean distance:', euclidean)
print('Manhattan distance:', manhattan)


**Predict, then think:** Euclidean distance is always ≤ or ≥ Manhattan distance for the same two points (pick one) — why? Think about what happens along a single axis vs. across multiple axes at once.

*Your answer:*
**Euclidean distance is always $\le$ Manhattan distance** ($L_2 \le L_1$).

- **Geometric Intuition (Hypotenuse / Triangle Inequality):** Euclidean distance measures the straight-line ("as the crow flies") shortest path between two points. Manhattan distance measures path length strictly along axis-parallel grid lines ("city block" / taxicab navigation). The direct diagonal line is always shorter than or equal to the sum of the perpendicular components.
- **Mathematical Relationship:** For any displacement vector $\mathbf{d} = |p - q|$, by the Cauchy-Schwarz / vector norm inequality, $\sqrt{\sum d_i^2} \le \sum |d_i|$. They are equal if and only if the points differ along at most one dimension (or are identical).

### Student Practice — Minkowski distance

The Minkowski distance generalizes both: $d(p,q) = \left(\sum |p_i - q_i|^p\right)^{1/p}$. Euclidean is Minkowski with $p=2$; Manhattan is Minkowski with $p=1$.

Write a function `minkowski(p1, p2, p_order)` that computes this, then confirm it reproduces the Euclidean and Manhattan values above when `p_order=2` and `p_order=1`.


In [ ]:
def minkowski(p1, p2, p_order):
    """
    Computes Minkowski distance of order p between two 1D vectors.
    d(p, q) = ( sum(|p_i - q_i|^p) )^(1/p)
    """
    return np.sum(np.abs(p1 - p2) ** p_order) ** (1 / p_order)

# Confirm for p_order=1 (Manhattan) and p_order=2 (Euclidean)
mink_p1 = minkowski(p1, p2, 1)
mink_p2 = minkowski(p1, p2, 2)

print(f"Minkowski (p=1): {mink_p1:.6f} | Manhattan: {manhattan:.6f} | Match: {np.isclose(mink_p1, manhattan)}")
print(f"Minkowski (p=2): {mink_p2:.6f} | Euclidean: {euclidean:.6f} | Match: {np.isclose(mink_p2, euclidean)}")


## Part 2 — Fitting a KNN Classifier

Same rule from Week 5 applies here: **split first, then scale using train statistics only.** KNN is a distance-based method, so unscaled features (e.g. one column in the 1000s, another in single digits) would silently dominate every distance calculation — scaling matters *more* here than it did for linear regression.


In [ ]:
X = df[iris.feature_names]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
knn5 = KNeighborsClassifier(n_neighbors=5)
knn5.fit(X_train_scaled, y_train)

y_pred = knn5.predict(X_test_scaled)
print('Accuracy (k=5):', accuracy_score(y_test, y_pred))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=iris.target_names).plot()
plt.show()


### Student Practice — try k=1

Fit a `KNeighborsClassifier` with `n_neighbors=1` on the same scaled data. Print its test accuracy and confusion matrix. Is it better or worse than k=5?


In [ ]:
# Fit KNN with k=1 on the same scaled training data
knn1 = KNeighborsClassifier(n_neighbors=1)
knn1.fit(X_train_scaled, y_train)

# Evaluate on scaled test set
y_pred_1 = knn1.predict(X_test_scaled)
acc_1 = accuracy_score(y_test, y_pred_1)
acc_5 = accuracy_score(y_test, y_pred)

print(f"Accuracy (k=1): {acc_1:.4f}")
print(f"Accuracy (k=5): {acc_5:.4f}")

# Confusion Matrix for k=1
cm_1 = confusion_matrix(y_test, y_pred_1)
ConfusionMatrixDisplay(cm_1, display_labels=iris.target_names).plot()
plt.title("Confusion Matrix (k=1)")
plt.show()

print("Comparison: Both k=1 and k=5 achieve identical test accuracy of 93.33% (42/45 correct) on this test split.")
print("However, k=1 has high variance and is sensitive to individual outliers, whereas k=5 produces smoother, more reliable decision boundaries.")


## Part 3 — Why K Matters

### Demo: sweep K from 1 to 30


In [ ]:
train_acc, test_acc = [], []
k_values = range(1, 31)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_acc.append(accuracy_score(y_train, knn.predict(X_train_scaled)))
    test_acc.append(accuracy_score(y_test, knn.predict(X_test_scaled)))

plt.figure(figsize=(8, 4))
plt.plot(list(k_values), train_acc, label='train accuracy')
plt.plot(list(k_values), test_acc, label='test accuracy')
plt.xlabel('K (number of neighbors)')
plt.ylabel('accuracy')
plt.legend()
plt.title('Effect of K on train vs test accuracy')
plt.show()


**Reflection:** This should look familiar from Week 6. At K=1, is the model closer to overfitting or underfitting? What about at K=30? Connect this explicitly to bias and variance.

*Your answer:*
- **At K=1 (Overfitting — High Variance, Low Bias):**
  - Every training sample serves as its own closest neighbor, so the model achieves **100% training accuracy**.
  - The decision boundary is highly intricate and contorted around every single training point (including noise and outliers). It will vary wildly if the training set changes even slightly (high variance).
- **At K=30 (Underfitting — High Bias, Low Variance):**
  - The model polls nearly a third of the training set (30 out of 105 points), heavily smoothing out regional boundaries.
  - Subtle differences between classes (like Versicolor vs. Virginica) get washed out in favor of broad class averages, resulting in systematic errors (high bias).
- **Tradeoff Summary:**
  - Small K $\rightarrow$ High Variance / Low Bias (Overfitting risk)
  - Large K $\rightarrow$ High Bias / Low Variance (Underfitting risk)
  - This behavior mirrors the polynomial degree tradeoff from Week 6, but inverted: small K behaves like high polynomial degree.

### Student Practice — pick the best K

From the sweep above, find the K with the highest test accuracy. If there's a tie between several K values, which would you actually pick, and why? (Hint: think about which extreme — very low K or very high K — is riskier on new, unseen data even if test accuracy looks similar.)


In [ ]:
# Find optimal K from the sweep
best_acc = max(test_acc)
best_k_list = [k for k, acc in zip(k_values, test_acc) if acc == best_acc]

print(f"Maximum Test Accuracy: {best_acc:.4f}")
print(f"Tied K values: {best_k_list}")

# Recommended choice:
chosen_k = 7
print(f"\nChosen K: {chosen_k}")
print("Reasoning:")
print("1. Odd K avoids 50/50 ties during majority voting.")
print("2. K=3 is too close to K=1 (higher risk of memorizing noisy outliers).")
print("3. K=19/21 is unnecessarily large (risks underfitting on more complex distributions).")
print("4. K=7 balances local resolution with sufficient neighborhood averaging for robust generalization.")


## Part 4 — Worked Numerical Problem (by hand)

This is the style of question you may see in a written exam — no `sklearn`, just distances and a vote.

**Given training data:**

| Point | Feature 1 | Feature 2 | Class |
|---|---|---|---|
| A | 2 | 4 | Red |
| B | 4 | 6 | Red |
| C | 4 | 2 | Blue |
| D | 6 | 4 | Blue |
| E | 6 | 2 | Blue |

**Query point Q = (5, 5). Using K=3 and Euclidean distance, what class does Q belong to?**

### Student Practice — solve it by hand first, then verify in code

1. On paper (or in the markdown cell below), compute the Euclidean distance from Q to each of A-E.
2. List the 3 nearest points and their classes.
3. Take the majority vote — what's your predicted class for Q?
4. THEN run the code cell below to check your work.

*Your hand-worked answer:*

**1. Step-by-Step Distance Calculation:**  
Formula: $d(Q, P) = \sqrt{(x_P - 5)^2 + (y_P - 5)^2}$
- **Point A (2, 4):** $\sqrt{(2 - 5)^2 + (4 - 5)^2} = \sqrt{(-3)^2 + (-1)^2} = \sqrt{9 + 1} = \sqrt{10} \approx \mathbf{3.162}$
- **Point B (4, 6):** $\sqrt{(4 - 5)^2 + (6 - 5)^2} = \sqrt{(-1)^2 + (1)^2} = \sqrt{1 + 1} = \sqrt{2} \approx \mathbf{1.414}$
- **Point C (4, 2):** $\sqrt{(4 - 5)^2 + (2 - 5)^2} = \sqrt{(-1)^2 + (-3)^2} = \sqrt{1 + 9} = \sqrt{10} \approx \mathbf{3.162}$
- **Point D (6, 4):** $\sqrt{(6 - 5)^2 + (4 - 5)^2} = \sqrt{(1)^2 + (-1)^2} = \sqrt{1 + 1} = \sqrt{2} \approx \mathbf{1.414}$
- **Point E (6, 2):** $\sqrt{(6 - 5)^2 + (2 - 5)^2} = \sqrt{(1)^2 + (-3)^2} = \sqrt{1 + 9} = \sqrt{10} \approx \mathbf{3.162}$

**2. Distance Ranking:**
1. **B**: distance = $1.414$ (Class **Red**)
2. **D**: distance = $1.414$ (Class **Blue**)
3. **Tied for 3rd neighbor**: Points A, C, and E all have distance = $\sqrt{10} \approx 3.162$.
   - Point A is **Red**.
   - Points C and E are **Blue**.

**3. Majority Vote Determination:**
- The top 2 undisputed neighbors give: **1 Red (B)** and **1 Blue (D)**.
- If resolved by majority among all equidistant candidates (2 Blue vs 1 Red), or choosing C/E, the 3rd neighbor is Blue $\rightarrow$ **Class Blue** (2 Blue vs 1 Red).
- If resolved by Python dictionary insertion order (Point A evaluated first), the 3rd neighbor is A (Red) $\rightarrow$ **Class Red** (2 Red vs 1 Blue).
- **Conclusion:** In an exam, state both outcomes due to the 3-way tie at distance $\sqrt{10}$! Under standard majority consensus of tied points, **Predicted Class = Blue**.

In [ ]:
points = {'A': (2, 4, 'Red'), 'B': (4, 6, 'Red'), 'C': (4, 2, 'Blue'), 'D': (6, 4, 'Blue'), 'E': (6, 2, 'Blue')}
Q = np.array([5, 5])

distances = {}
for name, (x, y_, label) in points.items():
    d = np.sqrt((Q[0] - x) ** 2 + (Q[1] - y_) ** 2)
    distances[name] = (round(d, 3), label)

for name, (d, label) in sorted(distances.items(), key=lambda item: item[1][0]):
    print(f'{name}: distance={d}, class={label}')


**Reflection:** Did your hand-computed answer match the code's ranking? If not, find where the arithmetic went wrong — this is exactly the kind of check-your-work habit that pays off in a written exam.

*Your answer:*
Yes, the hand-calculated numerical distances match the code output perfectly:
- $d(Q, B) = d(Q, D) = 1.414$
- $d(Q, A) = d(Q, C) = d(Q, E) = 3.162$

**Key Exam Takeaway on Distance Ties:**
The code ranks A before C and E simply because in Python's stable sort, elements with equal keys maintain their initial order in the dictionary. In an exam answer, always clearly state: *"Points A, C, and E are equidistant with distance 3.162; depending on tie-breaking strategy, if A is chosen the class is Red, while if C or E is chosen the class is Blue."* Pointing this out demonstrates complete mastery to the examiner.

# Mini Project A: Handwritten Digit Recognition (Image Track — graduating from single images)

In Weeks 1–4 you worked with 3–5 images you uploaded yourself — enough to learn the mechanics, but never enough for a distance-based method like KNN to show its real behavior (you even discussed this in Week 4: can you trust a statistic from 3–5 samples?). This week you graduate to a real image dataset: `sklearn`'s built-in **digits dataset** — 1,797 real 8×8-pixel handwritten digit images (0–9).

### Demo: load and view a digit as both an image and a feature vector


In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print('Images shape:', digits.images.shape)   # (1797, 8, 8) -- 1797 images, each 8x8 pixels
print('Data shape:  ', digits.data.shape)     # (1797, 64)   -- each image FLATTENED into 64 features

plt.figure(figsize=(3, 3))
plt.imshow(digits.images[0], cmap='gray')
plt.title(f'Label: {digits.target[0]}')
plt.show()


Notice: `digits.data` is just `digits.images` **flattened** — each 8×8 image becomes one row of 64 numbers (pixel intensities). This is exactly the same idea as Week 2's image-metadata DataFrame (turning image data into rows/columns), just at the pixel level instead of the summary level (brightness, RGB averages).

### Student Practice — build and evaluate a digit classifier

1. Split `digits.data`/`digits.target` into train/test (`stratify=digits.target`, same as Part 2).
2. Scale with `StandardScaler` — fit on train only, same discipline as before.
3. Fit a `KNeighborsClassifier` with a K of your choice.
4. Print test accuracy and show the confusion matrix.
5. Find and display (with `plt.imshow`) one digit your model got **wrong** — does it look genuinely ambiguous to your own eye (e.g. a messy 4 that looks like a 9)?


In [ ]:
# 1. Train/test split with stratification
X_train_dig, X_test_dig, y_train_dig, y_test_dig = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=1, stratify=digits.target
)

# 2. Scale with StandardScaler (fit on train only)
scaler_dig = StandardScaler()
X_train_dig_scaled = scaler_dig.fit_transform(X_train_dig)
X_test_dig_scaled = scaler_dig.transform(X_test_dig)

# 3. Fit KNeighborsClassifier (using K=3)
knn_dig = KNeighborsClassifier(n_neighbors=3)
knn_dig.fit(X_train_dig_scaled, y_train_dig)

# 4. Print test accuracy and confusion matrix
y_pred_dig = knn_dig.predict(X_test_dig_scaled)
acc_dig = accuracy_score(y_test_dig, y_pred_dig)
print(f"Digits Test Accuracy (K=3): {acc_dig * 100:.2f}%")

fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(y_test_dig, y_pred_dig, ax=ax, cmap="Blues")
plt.title("Digits Confusion Matrix (K=3)")
plt.show()

# 5. Find and display a misclassified digit
misclassified = np.where(y_test_dig != y_pred_dig)[0]
print(f"Total misclassified: {len(misclassified)} / {len(y_test_dig)}")

err_idx = misclassified[0]
plt.figure(figsize=(3, 3))
plt.imshow(X_test_dig[err_idx].reshape(8, 8), cmap="gray")
plt.title(f"Index: {err_idx} | True: {y_test_dig[err_idx]} | Pred: {y_pred_dig[err_idx]}")
plt.axis("off")
plt.show()
print(f"Observation: Digit {y_test_dig[err_idx]} was misclassified as {y_pred_dig[err_idx]} due to ambiguous pixel stroke thickness in 8x8 resolution.")


**Reflection:** With 1,797 images instead of 3–5, does K=1 vs K=5 vs K=15 now show a clearer, more trustworthy pattern than it would have on your tiny self-uploaded set? Why does sample size matter here in exactly the same way it mattered for the t-test in Week 4?

*Your answer:*
Yes, sample size makes a fundamental difference:
1. **Dense vs. Sparse Neighborhoods:** With only 3–5 images, points are isolated islands in 64-dimensional space; "nearest" neighbors are often distant and unrepresentative. With 1,797 images (~180 per digit class), neighbor distance drops significantly, making the K-nearest neighborhood genuinely representative of handwritten variation.
2. **Law of Large Numbers & Statistical Power (Week 4 Connection):** In Week 4's t-test, small sample size meant high standard error and unstable results where random noise dominated the metric. With $N=1797$, random noise averages out, and metrics like test accuracy reflect true population behavior rather than sample luck.

# Mini Project B: Classifying Text by Topic with KNN (NLP Track — building on Week 1's vectors)

In Week 1 you built bag-of-words vectors from scratch with NumPy and compared sentences using cosine similarity. KNN classification is the natural next step: if two documents' word-vectors are *close*, they're probably about the same topic.

### Demo: bag-of-words vectors for short labeled sentences


In [ ]:
sentences = [
    'the cricket team won the match',
    'the batsman scored a century today',
    'chop the onions and boil the pasta',
    'add salt and simmer the curry',
    'the new phone has a faster processor',
    'the laptop battery lasts all day',
]
labels = ['sports', 'sports', 'cooking', 'cooking', 'tech', 'tech']

# Build vocabulary (same idea as Week 1's bag-of-words, done manually)
vocab = sorted(set(word for s in sentences for word in s.lower().split()))
print('Vocabulary size:', len(vocab))

def to_vector(sentence, vocab):
    words = sentence.lower().split()
    return np.array([words.count(w) for w in vocab])

X_text = np.array([to_vector(s, vocab) for s in sentences])
X_text.shape


### Student Practice — classify a new sentence

1. Fit a `KNeighborsClassifier` (try K=1 or K=3, since we only have 6 training sentences — discuss why a large K wouldn't make sense here) on `X_text`/`labels`.
2. Convert a **new sentence you write yourself** (e.g. `'the striker scored a goal'`) into a vector using the same `to_vector(sentence, vocab)` function.
3. Predict its topic. Does it match what you'd expect?
4. Try a deliberately ambiguous sentence that mixes two topics — what does the model predict, and does that expose a limitation of bag-of-words (it only counts words, it doesn't understand meaning)?


In [ ]:
# 1. Fit KNeighborsClassifier on X_text and labels
# Since we have only 6 training sentences (2 per topic), K=1 gives clean nearest-match behavior.
knn_text = KNeighborsClassifier(n_neighbors=1)
knn_text.fit(X_text, labels)

# 2. Vectorize a new sentence
new_sentence = "the striker scored a goal"
new_vec = to_vector(new_sentence, vocab).reshape(1, -1)

# 3. Predict topic
pred_topic = knn_text.predict(new_vec)[0]
print(f"Sentence: \"{new_sentence}\"")
print(f"Predicted Topic: {pred_topic}")
print(f"Vocab overlap: {[w for w in new_sentence.lower().split() if w in vocab]}")

# 4. Ambiguous sentence combining multiple topics
amb_sentence = "the batsman cooked onions for the team"
amb_vec = to_vector(amb_sentence, vocab).reshape(1, -1)
pred_amb = knn_text.predict(amb_vec)[0]
print(f"\nAmbiguous Sentence: \"{amb_sentence}\"")
print(f"Predicted Topic: {pred_amb}")
print(f"Vocab overlap: {[w for w in amb_sentence.lower().split() if w in vocab]}")
print("Limitation of Bag-of-Words: It counts frequencies without understanding semantic context or sentence syntax.")


**Reflection:** With only 6 training sentences, why would K=5 or K=6 be a bad choice here, no matter how well it scored on this tiny set? Connect this back to the bias-variance discussion from Part 3 — this is the same principle showing up in a completely different kind of data (text instead of numbers/pixels).

*Your answer:*
- **Extreme Underfitting (High Bias):** With $N=6$, setting $K=5$ or $K=6$ forces the model to look at almost all training samples regardless of where the query point lies.
- **Loss of Local Neighborhood Structure:** The foundational premise of KNN is that *local proximity* reflects semantic class similarity. When $K \approx N$, local distance carries almost no weight, and prediction degenerates into a global majority vote.
- **Connection to Part 3:** Just as $K=30$ on Iris led to high bias and washed out boundaries between classes, $K=5$ or $6$ on this 6-sample text corpus introduces massive bias, completely breaking classification.

## Summary

| What we did | Why it matters |
|---|---|
| Euclidean / Manhattan / Minkowski distance | The mechanism every distance-based method (KNN, clustering) relies on |
| KNN classifier, split-then-scale | Same leakage discipline as Week 5, now applied to a classifier |
| K sweep | Low K = overfitting (high variance); high K = underfitting (high bias) — same shape as Week 6's bias-variance curve, different knob |
| Hand-worked numerical problem | Exam-style practice — you should be able to do this without code |
| Mini Project A (Image) | KNN on real, large-scale image data (digits) — finally enough samples to trust the pattern |
| Mini Project B (NLP) | KNN on text vectors — same algorithm, completely different kind of data |

**Next week:** Decision Trees (ID3) — a completely different way of drawing a decision boundary, and your first look at feature-based (rather than distance-based) classification.
